In [2]:
# extract_csv_kata.py
# Baca gambar dari folder dataset Roboflow
# Ekstrak landmark MediaPipe → simpan ke CSV

import cv2
import mediapipe as mp
import csv
import os
import numpy as np

# ================================
# SETUP MEDIAPIPE bisa deteksi tangan kiri & kanan,
# wajah, dan pose tubuh sekaligus dalam 1 proses
# ================================
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(
    static_image_mode=True,  # mode gambar, bukan video
    min_detection_confidence=0.5
)

# Titik-titik wajah yang dipilih (bukan semua, hanya yang penting)
# 1=hidung, 33=mata kiri, 61=mulut kiri, 199=dagu, 263=mata kanan
FACE_POINTS = [1, 33, 61, 199, 263]

# Titik-titik pose tubuh yang dipilih
# 11=bahu kiri, 12=bahu kanan, 13=siku kiri, 14=siku kanan, 15=pergelangan tangan kiri
POSE_POINTS = [11, 12, 13, 14, 15]

# ================================
# PATH DATASET
# ================================
DATASET_PATH = "dataset_kata"  
SPLITS = ["train", "valid", "test"]
OUTPUT_CSV = "data_kata2.csv"

# ================================
# FUNGSI EKSTRAK FITUR
# ================================
def extract_features(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None

    # Konversi BGR → RGB karena MediaPipe butuh format RGB
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = holistic.process(rgb)

    data = []

    def normalize_landmarks(landmarks): # merapikan posisi titik tangan supaya konsisten.
        pts = np.array([[lm.x, lm.y] for lm in landmarks])

        # RELATIVE (pakai wrist sebagai pusat)
        base = pts[0]
        pts = pts - base

        # SCALE (biar tidak tergantung jarak kamera)
        max_val = np.max(np.abs(pts))
        if max_val != 0:
            pts = pts / max_val

        return pts.flatten().tolist()

    def pairwise_distances(landmarks):
        # ambil beberapa jarak penting 
        pairs = [(4,8), (8,12), (12,16), (16,20)]
        dist = []
        for a,b in pairs:
            dx = landmarks[a].x - landmarks[b].x
            dy = landmarks[a].y - landmarks[b].y
            dist.append((dx**2 + dy**2)**0.5)
        return dist

    # =========================
    # LEFT HAND
    # =========================
    if result.left_hand_landmarks:
        left = result.left_hand_landmarks.landmark
        data += normalize_landmarks(left)
        data += pairwise_distances(left)
    else:
        data += [0.0] * (42 + 4)

    # =========================
    # RIGHT HAND
    # =========================
    if result.right_hand_landmarks:
        right = result.right_hand_landmarks.landmark
        data += normalize_landmarks(right)
        data += pairwise_distances(right)
    else:
        data += [0.0] * (42 + 4)

    # =========================
    # FACE (RELATIVE ke hidung)
    # =========================
    if result.face_landmarks:
        face = result.face_landmarks.landmark
        base = np.array([face[1].x, face[1].y]) # Hidung dijadikan patokan supaya posisi wajah stabil.

        for i in FACE_POINTS:
            lm = face[i]
            pt = np.array([lm.x, lm.y]) - base
            data += pt.tolist()
    else:
        data += [0.0] * 10

    # =========================
    # POSE (RELATIVE ke bahu)
    # =========================
    if result.pose_landmarks:
        pose = result.pose_landmarks.landmark
        base = np.array([pose[11].x, pose[11].y])  # Gunakan bahu kiri sebagai pusat biar posisi tubuh stabil

        for i in POSE_POINTS:
            lm = pose[i]
            pt = np.array([lm.x, lm.y]) - base
            data += pt.tolist()
    else:
        data += [0.0] * 10

    # =========================
    # CROSS FEATURES 
    # Menghitung jarak antara tangan dan wajah
    # =========================
    extra = []

    def dist_xy(a, b):
        return ((a.x - b.x)**2 + (a.y - b.y)**2) ** 0.5 #Ini rumus jarak biasa, buat tahu seberapa dekat dua titik.

    if result.right_hand_landmarks and result.face_landmarks:
        hand = result.right_hand_landmarks.landmark
        face = result.face_landmarks.landmark

        # Hitung jarak penting : Ini buat tahu posisi tangan terhadap wajah, misalnya dekat mulut atau tidak.”
        extra.append(dist_xy(hand[8], face[1]))   # telunjuk → hidung
        extra.append(dist_xy(hand[4], face[1]))   # ibu jari → hidung
    else:
        extra += [0.0, 0.0]

    data += extra  

    return data

# ================================
# PROSES SEMUA GAMBAR
# ================================
rows = []
label_counts = {}

for split in SPLITS:
    images_path = os.path.join(DATASET_PATH, split, "images")

    if not os.path.exists(images_path):
        print(f"Folder tidak ditemukan: {images_path}")
        continue

    # Ambil nama kelas dari file .txt label
    labels_path = os.path.join(DATASET_PATH, split, "labels")

    for img_file in os.listdir(images_path):
        if not img_file.endswith(('.jpg', '.jpeg', '.png')):
            continue

        img_path = os.path.join(images_path, img_file)

        # Ambil label dari file .txt : Cocokkan gambar dengan file labelnya (.txt)
        label_file = os.path.splitext(img_file)[0] + ".txt"
        label_path = os.path.join(labels_path, label_file)

        if not os.path.exists(label_path):
            continue

        # Baca kelas dari baris pertama file label
        with open(label_path, 'r') as f:
            line = f.readline().strip()
            if not line:
                continue
            class_id = int(line.split()[0])

        # Ekstrak fitur MediaPipe
        features = extract_features(img_path)

        if features is None:
            print(f"Skip (landmark tidak terdeteksi): {img_file}")
            continue

        rows.append(features + [class_id])

        label_counts[class_id] = label_counts.get(class_id, 0) + 1
        print(f"OK: {img_file} → kelas {class_id}")

# ================================
# SIMPAN KE CSV
# ================================
header = [f"f{i}" for i in range(114)] + ["label"]

with open(OUTPUT_CSV, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rows)

print(f"\n✅ Selesai! Total data: {len(rows)}")
print(f"📄 Tersimpan di: {OUTPUT_CSV}")
print(f"📊 Distribusi kelas: {label_counts}")

OK: berdoa-00921212-3137-11ef-8a82-9a8ad2a22026_jpg.rf.695899bf763a658f3fd8aa3d215e470e.jpg → kelas 0
OK: berdoa-00921212-3137-11ef-8a82-9a8ad2a22026_jpg.rf.6eb036dff0bdb334dc880d33ef16f90d.jpg → kelas 0
OK: berdoa-00921212-3137-11ef-8a82-9a8ad2a22026_jpg.rf.ac7f7eabe1da41655bec6d36c776c26a.jpg → kelas 0
OK: berdoa-04de8ed6-3136-11ef-8c2a-9a8ad2a22026_jpg.rf.11358207e039dec2347a042e8c0fc1b0.jpg → kelas 0
OK: berdoa-04de8ed6-3136-11ef-8c2a-9a8ad2a22026_jpg.rf.1372f358a3dd63ed8f9c48c8484aa86b.jpg → kelas 0
OK: berdoa-04de8ed6-3136-11ef-8c2a-9a8ad2a22026_jpg.rf.9e6b6f94d317ff2265eff602c09579a6.jpg → kelas 0
OK: berdoa-0567ce8b-3137-11ef-9ab5-9a8ad2a22026_jpg.rf.3eed7eabee8378820ad645c546707459.jpg → kelas 0
OK: berdoa-0567ce8b-3137-11ef-9ab5-9a8ad2a22026_jpg.rf.7dffbaa9198444b08d3a10287b60151d.jpg → kelas 0
OK: berdoa-0567ce8b-3137-11ef-9ab5-9a8ad2a22026_jpg.rf.8e9cc4060cb140253de65a924e18be09.jpg → kelas 0
OK: berdoa-1369d5e3-3136-11ef-a8b4-9a8ad2a22026_jpg.rf.679e4ec3b0ead03d68a7ecb0e16

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

# ================================
# LOAD CSV
# ================================
df = pd.read_csv("data_kata2.csv")

X = df.drop("label", axis=1).values
y = df["label"].values

print(f"Total data: {len(df)}")
print(f"Jumlah kelas: {len(set(y))}")
print(f"Jumlah fitur: {X.shape[1]}")  


# ================================
# SPLIT DATA
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ================================
# TRAINING
# ================================
print("\nTraining Random Forest (Optimized)...")

model = RandomForestClassifier(
    n_estimators=200,        
    max_depth=None,          
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced", 
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# ================================
# EVALUASI
# ================================
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"\n  Akurasi: {acc * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ================================
# SIMPAN MODEL
# ================================
joblib.dump(model, "model_kata.pkl")
print("\n Model tersimpan: model_kata.pkl")

Total data: 1190
Jumlah kelas: 7
Jumlah fitur: 114

Training Random Forest (Optimized)...

  Akurasi: 94.12%

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.97      0.90        34
           1       1.00      0.94      0.97        33
           2       0.94      0.91      0.93        34
           3       0.91      0.91      0.91        34
           4       1.00      1.00      1.00        33
           5       0.91      0.89      0.90        35
           6       1.00      0.97      0.99        35

    accuracy                           0.94       238
   macro avg       0.94      0.94      0.94       238
weighted avg       0.94      0.94      0.94       238


 Model tersimpan: model_kata.pkl
